# 03 Final SuperPro output inventory transformation (20260625)

This notebook is the final transformation step for the SuperPro output inventory used in the thesis LCA and optimisation workflow.

The purpose is to keep a complete, auditable output-side dataset from the SuperPro SR and ICR reports without losing quantities or hiding duplicate accounting bases. The final sheet therefore contains several different views of the same process outputs:

1. **SR stream component rows**: composition of each `Destination = OUTPUT` stream.
2. **SR stream total rows**: the routable stream totals used for industrial symbiosis and waste routing.
3. **SR overall component balance rows**: the whole-process component output balance from SuperPro.
4. **SR reconciliation rows**: the difference between the overall component balance and the sum of explicit stream component rows. These rows are not physical streams. They are included so that nothing from the SR overall balance disappears.
5. **ICR SUT end-of-life rows**: used single-use consumables from the ICR, treated as end-of-life output candidates. These remain in item units until kg/item plastic mapping is completed.

The table is a **master output inventory**, not a table to sum directly. The `inventory_role`, `accounting_basis`, and `include_in_is_inventory` columns define which rows should be used for routing, validation, LCA treatment, or metadata.


In [ ]:
from pathlib import Path
from datetime import date
import re
import math

import pandas as pd
import xlrd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 180)

RUN_DATE = date.today().strftime('%Y%m%d')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'rawSP_reports'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f'Supplied raw SuperPro reports directory is missing: {RAW_DIR}')

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)


## 1. Locate the SR and ICR reports

The SR gives process streams and the overall component balance. The ICR gives single-use consumable usage. I do not use the ICR utility tables here because this sheet is only the output inventory transformation. Utilities belong in the LCA operating inventory.


In [ ]:
def find_report(patterns, label):
    matches = []
    for pattern in patterns:
        matches.extend(sorted(RAW_DIR.glob(pattern)))
    matches = list(dict.fromkeys(matches))
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one supplied {label} in {RAW_DIR}; found {matches}')
    return matches[0]

SR_PATH = find_report(['*SR*.xls', '*SR*.xlsx'], 'SuperPro SR report')
ICR_PATH = find_report(['*ICR*.xls', '*ICR*.xlsx'], 'SuperPro ICR report')

print('Using SR:', SR_PATH)
print('Using ICR:', ICR_PATH)


## 2. Helpers for SuperPro exported reports

The SuperPro Excel exports are not normal databases. Each report is a single long worksheet containing many stacked tables. I parse them from table boundaries and row labels rather than selecting rows manually.


In [ ]:
def cell_to_text(x):
    if x is None:
        return ''
    if isinstance(x, float):
        if math.isnan(x):
            return ''
        if x.is_integer():
            return str(int(x))
        return str(x)
    return str(x).strip()


def clean_text(x):
    s = cell_to_text(x).replace(' ', ' ')
    s = re.sub(r'\s+', ' ', s)
    return s.strip()


def to_number(x):
    s = clean_text(x)
    if s == '' or s.lower() in {'nan', 'none', 'unknown', '(none)'}:
        return None
    s = s.replace(',', '').replace('−', '-')
    try:
        return float(s)
    except ValueError:
        return None


def load_first_sheet(path):
    book = xlrd.open_workbook(str(path), formatting_info=False)
    sheet = book.sheet_by_index(0)
    rows = []
    for r in range(sheet.nrows):
        rows.append([cell_to_text(sheet.cell_value(r, c)) for c in range(sheet.ncols)])
    return pd.DataFrame(rows), sheet.name


def find_row_containing(df, text, start=0, end=None, col=None):
    if end is None:
        end = len(df)
    needle = text.lower()
    for r in range(start, min(end, len(df))):
        cols = [col] if col is not None else range(df.shape[1])
        for c in cols:
            if c < df.shape[1] and needle in clean_text(df.iloc[r, c]).lower():
                return r
    return None


def amount_per_kg_mp(amount):
    if amount is None or batch_size_kg_mp in [None, 0]:
        return None
    return amount / batch_size_kg_mp

sr_df, sr_sheet_name = load_first_sheet(SR_PATH)
icr_df, icr_sheet_name = load_first_sheet(ICR_PATH)

print('SR sheet:', sr_sheet_name, sr_df.shape)
print('ICR sheet:', icr_sheet_name, icr_df.shape)


## 3. Process basis

The number of batches per year and kg mAb per batch are used for annualised quantities and normalisation per kg mAb product. The kg mAb basis is taken from the SR overall process data, not recalculated manually.


In [ ]:
def extract_sr_process_basis():
    start = find_row_containing(sr_df, '1. OVERALL PROCESS DATA')
    end = find_row_containing(sr_df, '2.1 STARTING MATERIAL REQUIREMENTS', start=start + 1)
    records = []

    if end is None:
        end = start + 30
    for r in range(start + 1, end):
        label = clean_text(sr_df.iloc[r, 1]) if sr_df.shape[1] > 1 else ''
        if label == '':
            continue
        value = None
        unit = ''
        for c in range(2, sr_df.shape[1]):
            parsed = to_number(sr_df.iloc[r, c])
            if parsed is not None:
                value = parsed
                for ucol in range(c + 1, sr_df.shape[1]):
                    possible_unit = clean_text(sr_df.iloc[r, ucol])
                    if possible_unit and to_number(possible_unit) is None:
                        unit = possible_unit
                        break
                break
        if value is not None:
            records.append({'source_row': r, 'parameter': label, 'value': value, 'unit': unit})
    return pd.DataFrame(records)

process_basis_df = extract_sr_process_basis()
display(process_basis_df)

batch_size_kg_mp = float(process_basis_df.loc[process_basis_df['parameter'].str.contains('Batch Size', case=False, na=False), 'value'].iloc[0])
batches_per_year = float(process_basis_df.loc[process_basis_df['parameter'].str.contains('Number of Batches', case=False, na=False), 'value'].iloc[0])

print('batch_size_kg_mp:', batch_size_kg_mp)
print('batches_per_year:', batches_per_year)


## 4. Procedure mapping retained from the flowsheet review

This mapping keeps the section and unit-operation labels already checked against the flowsheet. I use it directly instead of remapping from keywords again.


In [ ]:
procedure_mapping_records = [
    # Upstream: inoculum preparation
    {'source_procedure': 'P-01', 'broad_process_area': 'upstream', 'superpro_section': 'Inoculum Prep (Upstream)', 'unit_operation': 'Ampule', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-02', 'broad_process_area': 'upstream', 'superpro_section': 'Inoculum Prep (Upstream)', 'unit_operation': 'T-Flask', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-03', 'broad_process_area': 'upstream', 'superpro_section': 'Inoculum Prep (Upstream)', 'unit_operation': 'Roller Bottle', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-04', 'broad_process_area': 'upstream', 'superpro_section': 'Inoculum Prep (Upstream)', 'unit_operation': 'Disposable Bioreactor (100 L)', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-05', 'broad_process_area': 'upstream', 'superpro_section': 'Inoculum Prep (Upstream)', 'unit_operation': 'Disposable Bioreactor (600 L)', 'area_mapping_basis': 'flowsheet_confirmed'},

    # Upstream: cell culture / media preparation
    {'source_procedure': 'P-06', 'broad_process_area': 'upstream', 'superpro_section': 'Cell Culture (Upstream)', 'unit_operation': 'Media Prep Tank', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-07', 'broad_process_area': 'upstream', 'superpro_section': 'Cell Culture (Upstream)', 'unit_operation': 'Sterile Filtration', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-08', 'broad_process_area': 'upstream', 'superpro_section': 'Cell Culture (Upstream)', 'unit_operation': 'Media Hold Tank', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-09', 'broad_process_area': 'upstream', 'superpro_section': 'Cell Culture (Upstream)', 'unit_operation': 'Cell Culture Bioreactor', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-10', 'broad_process_area': 'upstream', 'superpro_section': 'Cell Culture (Upstream)', 'unit_operation': 'Mixing', 'area_mapping_basis': 'flowsheet_confirmed'},

    # Downstream recovery and purification
    {'source_procedure': 'P-11', 'broad_process_area': 'downstream', 'superpro_section': 'Primary Recovery (Downstream)', 'unit_operation': 'Centrifugation', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-13', 'broad_process_area': 'downstream', 'superpro_section': 'Primary Recovery (Downstream)', 'unit_operation': 'Clarification / Dead-End Filtration', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-14', 'broad_process_area': 'downstream', 'superpro_section': 'Primary Recovery (Downstream)', 'unit_operation': 'Surge Tank', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-15', 'broad_process_area': 'downstream', 'superpro_section': 'Protein A (Downstream)', 'unit_operation': 'Protein A Chromatography', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-17', 'broad_process_area': 'downstream', 'superpro_section': 'Protein A (Downstream)', 'unit_operation': 'C-1 Feed Vessel', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-18', 'broad_process_area': 'downstream', 'superpro_section': 'Protein A (Downstream)', 'unit_operation': 'Protein A Chromatography', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-19', 'broad_process_area': 'downstream', 'superpro_section': 'Diafiltration / Viral Inactivation (Downstream)', 'unit_operation': 'Diafiltration Feed Tank', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-20', 'broad_process_area': 'downstream', 'superpro_section': 'Diafiltration (Downstream)', 'unit_operation': 'Diafiltration', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-21', 'broad_process_area': 'downstream', 'superpro_section': 'HIC Chromatography (Downstream)', 'unit_operation': 'HIC Chromatography', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-22', 'broad_process_area': 'downstream', 'superpro_section': 'HIC Chromatography (Downstream)', 'unit_operation': 'C2 Eluant Collection', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-23', 'broad_process_area': 'downstream', 'superpro_section': 'Membrane Adsorption / Diafiltration (Downstream)', 'unit_operation': 'Diafiltration', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-24', 'broad_process_area': 'downstream', 'superpro_section': 'Viral Inactivation (Downstream)', 'unit_operation': 'Inactivation Vessel', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-12', 'broad_process_area': 'downstream', 'superpro_section': 'Membrane Adsorption (Downstream)', 'unit_operation': 'Membrane Adsorption', 'area_mapping_basis': 'flowsheet_confirmed'},

    # Downstream buffer preparation and storage. These support downstream purification, so broad area is downstream.
    {'source_procedure': 'P-18Aa', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / Storage Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-18Ba', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Storage / EQ Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-18Ca', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / Rinse Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-18Da', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / Elution Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-18Ea', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / Wash Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-18Fa', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / 0.5 M NaOH', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-20a', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / C-2 Prep Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-21Aa', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / C-2 Regen Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-21Ba', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / C-2 EQ and Chase Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-23a', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / PBS', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-12Aa', 'broad_process_area': 'downstream', 'superpro_section': 'Buffer Prep and Hold (Downstream)', 'unit_operation': 'Buffer Prep / C-3 EQ-Wash Buffer', 'area_mapping_basis': 'flowsheet_confirmed'},

    # Filling / final product handling
    {'source_procedure': 'P-25', 'broad_process_area': 'formulation_fill', 'superpro_section': 'Filling', 'unit_operation': 'Sterile 0.22 Filtration', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-26', 'broad_process_area': 'formulation_fill', 'superpro_section': 'Filling', 'unit_operation': '20 L Bag', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-27', 'broad_process_area': 'formulation_fill', 'superpro_section': 'Filling', 'unit_operation': 'Freezer / Product', 'area_mapping_basis': 'flowsheet_confirmed'},
    {'source_procedure': 'P-28', 'broad_process_area': 'formulation_fill', 'superpro_section': 'Filling', 'unit_operation': 'Final Collection / Storage', 'area_mapping_basis': 'flowsheet_confirmed'},
]

procedure_mapping_df = pd.DataFrame(procedure_mapping_records).drop_duplicates('source_procedure')
procedure_mapping = procedure_mapping_df.set_index('source_procedure').to_dict('index')

def map_procedure(proc):
    proc = clean_text(proc)
    if proc in procedure_mapping:
        return procedure_mapping[proc]
    return {
        'broad_process_area': 'review_needed',
        'superpro_section': '',
        'unit_operation': '',
        'area_mapping_basis': 'not_mapped',
    }

display(procedure_mapping_df)


## 5. Extract all SR `Destination = OUTPUT` streams

Every stream with `Destination = OUTPUT` is retained. I keep both the component rows and the stream total rows because they answer different questions:

- component rows explain what the stream contains;
- total rows define the routeable stream mass and volume.

The table must not be summed across both bases.


In [ ]:
def classify_stream_output_type(stream_name):
    s = clean_text(stream_name).lower()
    if s == 'product' or 'product' in s:
        return 'product_output'
    if 'vent' in s:
        return 'vent_output'
    if 'waste' in s:
        return 'waste_output'
    return 'process_output'


def inventory_role_for_stream(record_type, stream_name):
    output_type = classify_stream_output_type(stream_name)
    if output_type == 'product_output':
        return 'reference_product' if record_type == 'output_stream_total' else 'reference_product_composition'
    if record_type == 'output_stream_total':
        return 'active_is_routing_mass_and_lca_treatment_candidate'
    return 'composition_metadata_only'


def include_is_for_stream(record_type, stream_name):
    output_type = classify_stream_output_type(stream_name)
    if output_type == 'product_output':
        return 'no_reference_product'
    if record_type == 'output_stream_total':
        return 'yes_active_routing_mass'
    return 'no_metadata_only'


def extract_sr_output_streams():
    block_starts = [r for r in range(len(sr_df)) if clean_text(sr_df.iloc[r, 1]) == 'Stream Name']
    component_balance_start = find_row_containing(sr_df, '4. OVERALL COMPONENT BALANCE', col=1)

    total_rows = []
    component_rows = []

    for block_index, start in enumerate(block_starts):
        block_end = block_starts[block_index + 1] if block_index + 1 < len(block_starts) else component_balance_start
        component_header = None
        for r in range(start, block_end):
            if clean_text(sr_df.iloc[r, 1]) == 'Component Flowrates (kg/batch)':
                component_header = r
                break
        if component_header is None:
            continue

        stream_columns = []
        for c in range(2, sr_df.shape[1]):
            stream_name = clean_text(sr_df.iloc[start, c])
            if stream_name != '':
                stream_columns.append((c, stream_name))

        for c, stream_name in stream_columns:
            source = clean_text(sr_df.iloc[start + 1, c])
            destination = clean_text(sr_df.iloc[start + 2, c])
            if destination.upper() != 'OUTPUT':
                continue

            proc_map = map_procedure(source)
            output_type = classify_stream_output_type(stream_name)

            stream_total_kg = None
            stream_total_l = None
            stream_components = []

            r = component_header + 1
            while r < block_end:
                component = clean_text(sr_df.iloc[r, 1])
                if component == '':
                    break
                amount = to_number(sr_df.iloc[r, c])
                if component == 'TOTAL (kg/batch)':
                    stream_total_kg = amount
                elif component == 'TOTAL (L/batch)':
                    stream_total_l = amount
                elif amount is not None and abs(amount) > 0:
                    stream_components.append({'component': component, 'amount_per_batch': amount, 'source_row': r})
                r += 1

            component_sum = sum(x['amount_per_batch'] for x in stream_components)
            total_difference = None if stream_total_kg is None else stream_total_kg - component_sum

            for component_record in stream_components:
                amount = component_record['amount_per_batch']
                component_rows.append({
                    'source_report': 'SR',
                    'source_table': '3. STREAM DETAILS',
                    'record_type': 'output_stream_component',
                    'inventory_role': inventory_role_for_stream('output_stream_component', stream_name),
                    'accounting_basis': 'stream_component_mass; do not add to stream total rows',
                    'broad_process_area': proc_map['broad_process_area'],
                    'superpro_section': proc_map['superpro_section'],
                    'unit_operation': proc_map['unit_operation'],
                    'area_mapping_basis': proc_map['area_mapping_basis'],
                    'source_procedure': source,
                    'destination': destination,
                    'flow_name_superpro': stream_name,
                    'output_type': output_type,
                    'component': component_record['component'],
                    'amount_per_batch': amount,
                    'unit_per_batch': 'kg/batch',
                    'amount_per_year': amount * batches_per_year,
                    'unit_per_year': 'kg/year',
                    'amount_per_kg_mp': amount_per_kg_mp(amount),
                    'unit_per_kg_mp': 'kg/kg MP',
                    'stream_volume_L_per_batch': None,
                    'stream_volume_L_per_year': None,
                    'stream_component_sum_kg_per_batch': component_sum,
                    'stream_total_minus_component_sum_kg_per_batch': total_difference,
                    'include_in_is_inventory': include_is_for_stream('output_stream_component', stream_name),
                    'mapping_status': 'composition metadata from explicit SR output stream',
                    'notes': 'Non-zero component in a SuperPro boundary stream where Destination = OUTPUT.',
                    'source_row': component_record['source_row'],
                    'source_column': c,
                })

            total_rows.append({
                'source_report': 'SR',
                'source_table': '3. STREAM DETAILS',
                'record_type': 'output_stream_total',
                'inventory_role': inventory_role_for_stream('output_stream_total', stream_name),
                'accounting_basis': 'stream_total; do not add to component rows',
                'broad_process_area': proc_map['broad_process_area'],
                'superpro_section': proc_map['superpro_section'],
                'unit_operation': proc_map['unit_operation'],
                'area_mapping_basis': proc_map['area_mapping_basis'],
                'source_procedure': source,
                'destination': destination,
                'flow_name_superpro': stream_name,
                'output_type': output_type,
                'component': 'TOTAL',
                'amount_per_batch': stream_total_kg,
                'unit_per_batch': 'kg/batch',
                'amount_per_year': stream_total_kg * batches_per_year if stream_total_kg is not None else None,
                'unit_per_year': 'kg/year',
                'amount_per_kg_mp': amount_per_kg_mp(stream_total_kg),
                'unit_per_kg_mp': 'kg/kg MP',
                'stream_volume_L_per_batch': stream_total_l,
                'stream_volume_L_per_year': stream_total_l * batches_per_year if stream_total_l is not None else None,
                'stream_component_sum_kg_per_batch': component_sum,
                'stream_total_minus_component_sum_kg_per_batch': total_difference,
                'include_in_is_inventory': include_is_for_stream('output_stream_total', stream_name),
                'mapping_status': 'explicit SR output stream total',
                'notes': 'Routeable stream total from SR. Use this basis for IS routing, excluding the final product stream.',
                'source_row': start,
                'source_column': c,
            })

    return pd.DataFrame(component_rows), pd.DataFrame(total_rows)

sr_output_stream_components_df, sr_output_stream_totals_df = extract_sr_output_streams()

print('Output stream component rows:', len(sr_output_stream_components_df))
print('Output stream total rows:', len(sr_output_stream_totals_df))
print('Total output stream mass kg/batch:', sr_output_stream_totals_df['amount_per_batch'].sum())
display(sr_output_stream_totals_df[['source_procedure','flow_name_superpro','broad_process_area','superpro_section','unit_operation','output_type','amount_per_batch','stream_volume_L_per_batch','stream_total_minus_component_sum_kg_per_batch']])


## 6. Extract the SR overall component output balance

The overall component output balance is retained because it is the most complete whole-process output balance. It is not the same accounting basis as the explicit stream totals.


In [ ]:
def extract_sr_overall_component_outputs(include_zero_components=True):
    start = find_row_containing(sr_df, '4. OVERALL COMPONENT BALANCE (kg/batch)')
    end = find_row_containing(sr_df, '5. EQUIPMENT CONTENTS', start=start + 1)
    header = find_row_containing(sr_df, 'COMPONENT', start=start, end=end, col=1)

    records = []
    for r in range(header + 1, end):
        component = clean_text(sr_df.iloc[r, 1])
        if component == '' or component.upper() == 'TOTAL' or component.startswith('NOTE:'):
            continue
        output_kg_per_batch = to_number(sr_df.iloc[r, 10])
        if output_kg_per_batch is None:
            continue
        if (not include_zero_components) and output_kg_per_batch == 0:
            continue
        records.append({
            'source_report': 'SR',
            'source_table': '4. OVERALL COMPONENT BALANCE (kg/batch)',
            'record_type': 'overall_component_output',
            'inventory_role': 'overall_balance_validation_or_lca_aggregate_basis',
            'accounting_basis': 'overall_component_balance; do not add to stream rows',
            'broad_process_area': 'overall_process',
            'superpro_section': 'Entire Process',
            'unit_operation': 'Entire Process',
            'area_mapping_basis': 'SR overall component balance',
            'source_procedure': '',
            'destination': 'OUTPUT',
            'flow_name_superpro': component,
            'output_type': 'overall_bulk_component_output',
            'component': component,
            'amount_per_batch': output_kg_per_batch,
            'unit_per_batch': 'kg/batch',
            'amount_per_year': output_kg_per_batch * batches_per_year,
            'unit_per_year': 'kg/year',
            'amount_per_kg_mp': amount_per_kg_mp(output_kg_per_batch),
            'unit_per_kg_mp': 'kg/kg MP',
            'stream_volume_L_per_batch': None,
            'stream_volume_L_per_year': None,
            'stream_component_sum_kg_per_batch': None,
            'stream_total_minus_component_sum_kg_per_batch': None,
            'include_in_is_inventory': 'no_validation_or_lca_aggregate_only',
            'mapping_status': 'overall output balance from SR',
            'notes': 'Whole-process component output balance. Use for reconciliation and possible aggregate LCA checks, not together with stream rows.',
            'source_row': r,
            'source_column': 10,
        })
    return pd.DataFrame(records)

sr_overall_component_outputs_df = extract_sr_overall_component_outputs(include_zero_components=False)
print('Overall component output rows:', len(sr_overall_component_outputs_df))
print('Overall component output mass kg/batch:', sr_overall_component_outputs_df['amount_per_batch'].sum())
display(sr_overall_component_outputs_df[['component','amount_per_batch','amount_per_year','amount_per_kg_mp']])


## 7. Reconcile stream components to the overall SR component output balance

This is the key consistency check. The explicit stream components do not explain the whole SR overall component output balance. Instead of dropping the difference, I create reconciliation rows. These are **not physical streams** and should not be routed directly, but they make the master output inventory complete against the SuperPro overall balance.


In [ ]:
def build_overall_vs_stream_reconciliation():
    stream_by_component = sr_output_stream_components_df.groupby('component', dropna=False)['amount_per_batch'].sum().rename('stream_component_sum_kg_per_batch')
    overall_by_component = sr_overall_component_outputs_df.groupby('component', dropna=False)['amount_per_batch'].sum().rename('overall_component_output_kg_per_batch')
    comp = pd.concat([overall_by_component, stream_by_component], axis=1).fillna(0).reset_index()
    comp['overall_minus_stream_components_kg_per_batch'] = comp['overall_component_output_kg_per_batch'] - comp['stream_component_sum_kg_per_batch']
    comp['overall_minus_stream_components_kg_per_year'] = comp['overall_minus_stream_components_kg_per_batch'] * batches_per_year
    comp['overall_minus_stream_components_kg_per_kg_mp'] = comp['overall_minus_stream_components_kg_per_batch'].apply(amount_per_kg_mp)
    return comp.sort_values('overall_minus_stream_components_kg_per_batch', key=lambda s: s.abs(), ascending=False)

reconciliation_df = build_overall_vs_stream_reconciliation()
display(reconciliation_df)

residual_rows = []
for _, row in reconciliation_df.iterrows():
    diff = row['overall_minus_stream_components_kg_per_batch']
    if abs(diff) <= 0.005:
        continue
    residual_rows.append({
        'source_report': 'SR',
        'source_table': '4. OVERALL COMPONENT BALANCE minus 3. STREAM DETAILS',
        'record_type': 'overall_minus_stream_detail_residual',
        'inventory_role': 'reconciliation_row_not_physical_stream',
        'accounting_basis': 'reconciliation_difference; do not route directly',
        'broad_process_area': 'overall_process',
        'superpro_section': 'Entire Process',
        'unit_operation': 'Overall balance reconciliation',
        'area_mapping_basis': 'calculated from SR overall balance and explicit output stream components',
        'source_procedure': '',
        'destination': 'OUTPUT_RECONCILIATION',
        'flow_name_superpro': row['component'],
        'output_type': 'overall_minus_stream_detail_residual',
        'component': row['component'],
        'amount_per_batch': diff,
        'unit_per_batch': 'kg/batch',
        'amount_per_year': diff * batches_per_year,
        'unit_per_year': 'kg/year',
        'amount_per_kg_mp': amount_per_kg_mp(diff),
        'unit_per_kg_mp': 'kg/kg MP',
        'stream_volume_L_per_batch': None,
        'stream_volume_L_per_year': None,
        'stream_component_sum_kg_per_batch': row['stream_component_sum_kg_per_batch'],
        'stream_total_minus_component_sum_kg_per_batch': None,
        'include_in_is_inventory': 'no_reconciliation_only',
        'mapping_status': 'overall balance not represented exactly by explicit stream component rows',
        'notes': 'Calculated as SR overall component output minus sum of explicit Destination=OUTPUT stream components. Positive values are output quantities present in the overall SR balance but not represented in explicit stream component rows; small +/-0.01 values are rounding.',
        'source_row': '',
        'source_column': '',
    })

sr_reconciliation_rows_df = pd.DataFrame(residual_rows)
print('Reconciliation rows:', len(sr_reconciliation_rows_df))
display(sr_reconciliation_rows_df[['component','amount_per_batch','amount_per_year','mapping_status','notes']])


## 8. Extract used SUT outputs from the ICR

Used SUT is an output material for the industrial symbiosis work, even though it appears as a consumable in the ICR. These rows remain in item units until kg/item and plastic composition are filled in. I keep all likely SUT plastic/filter items here and leave composition mapping for the next step.


In [ ]:
def parse_unit_from_header(header_text):
    match = re.search(r'\(([^()]*)\)', clean_text(header_text))
    return match.group(1) if match else ''


def broad_area_from_icr_section(section_name):
    s = clean_text(section_name).lower()
    if 'inoculum' in s or 'cell culture' in s:
        return 'upstream'
    if 'filling' in s or 'fill' in s:
        return 'formulation_fill'
    if 'buffer' in s or 'primary' in s or 'viral' in s or 'hic' in s or 'dft' in s or 'membrane' in s or 'chrom' in s:
        return 'downstream'
    return 'review_needed'


def classify_sut_material(name):
    s = clean_text(name).lower()
    if any(k in s for k in ['bag', 'cultibag', 'flask', 'tube', 'bottle']):
        return 'sut_plastic_bag_or_vessel'
    if any(k in s for k in ['cartridge', 'filter', 'membrane']):
        return 'sut_filter_or_membrane'
    if 'protein a' in s or 'resin' in s:
        return 'chromatography_resin_or_media_not_plastic'
    return 'review_needed'


def is_sut_candidate(name):
    return classify_sut_material(name) in {'sut_plastic_bag_or_vessel', 'sut_filter_or_membrane'}


def extract_icr_sut_by_section():
    start = find_row_containing(icr_df, '7.2 CONSUMABLES COST - BREAKDOWN BY SECTION')
    end = find_row_containing(icr_df, '7.3 CONSUMABLES COST - BREAKDOWN BY CONSUMABLE CATEGORY', start=start + 1)

    records = []
    current_section = None
    amount_unit_per_year = ''
    in_table = False

    for r in range(start + 1, end):
        label = clean_text(icr_df.iloc[r, 1])
        if label == '':
            continue

        if label == 'Consumable':
            amount_unit_per_year = parse_unit_from_header(icr_df.iloc[r, 14])
            in_table = True
            continue

        if label.upper() == 'TOTAL':
            in_table = False
            continue

        next_label = clean_text(icr_df.iloc[r + 1, 1]) if r + 1 < len(icr_df) else ''
        if next_label == 'Consumable':
            current_section = label
            in_table = False
            continue

        if in_table and current_section:
            amount_per_year = to_number(icr_df.iloc[r, 14])
            cost_per_year = to_number(icr_df.iloc[r, 21])
            if amount_per_year is None or not is_sut_candidate(label):
                continue
            records.append({
                'process_section': current_section,
                'broad_process_area': broad_area_from_icr_section(current_section),
                'consumable_name': label,
                'material_category': classify_sut_material(label),
                'amount_per_year': amount_per_year,
                'amount_unit_per_year': amount_unit_per_year,
                'amount_per_batch': amount_per_year / batches_per_year,
                'amount_unit_per_batch': amount_unit_per_year.replace('/year', '/batch'),
                'cost_per_year': cost_per_year,
                'cost_per_batch': cost_per_year / batches_per_year if cost_per_year is not None else None,
                'source_row': r,
            })
    return pd.DataFrame(records)

icr_sut_by_section_df = extract_icr_sut_by_section()

sut_group_cols = ['broad_process_area', 'process_section', 'consumable_name', 'material_category', 'amount_unit_per_year', 'amount_unit_per_batch']
icr_sut_aggregated_df = icr_sut_by_section_df.groupby(sut_group_cols, dropna=False).agg(
    amount_per_year=('amount_per_year', 'sum'),
    amount_per_batch=('amount_per_batch', 'sum'),
    cost_per_year=('cost_per_year', 'sum'),
    cost_per_batch=('cost_per_batch', 'sum'),
    source_rows=('source_row', lambda x: ';'.join(map(str, x))),
).reset_index()

print('SUT rows by ICR section:', len(icr_sut_by_section_df))
print('Aggregated SUT rows:', len(icr_sut_aggregated_df))
display(icr_sut_aggregated_df)


In [ ]:
def build_sut_output_rows(df):
    records = []
    for _, row in df.iterrows():
        amount_batch = row['amount_per_batch']
        records.append({
            'source_report': 'ICR',
            'source_table': '7.2 CONSUMABLES COST - BREAKDOWN BY SECTION',
            'record_type': 'sut_end_of_life_output',
            'inventory_role': 'sut_count_needs_mass_mapping',
            'accounting_basis': 'sut_usage; convert to kg plastic before LCA/IS mass balance',
            'broad_process_area': row['broad_process_area'],
            'superpro_section': row['process_section'],
            'unit_operation': 'SUT consumable use',
            'area_mapping_basis': 'ICR section name',
            'source_procedure': '',
            'destination': 'END_OF_LIFE_OUTPUT',
            'flow_name_superpro': row['consumable_name'],
            'output_type': row['material_category'],
            'component': 'mixed SUT plastic/material',
            'amount_per_batch': amount_batch,
            'unit_per_batch': row['amount_unit_per_batch'],
            'amount_per_year': row['amount_per_year'],
            'unit_per_year': row['amount_unit_per_year'],
            'amount_per_kg_mp': amount_per_kg_mp(amount_batch),
            'unit_per_kg_mp': row['amount_unit_per_batch'].replace('/batch', '/kg MP'),
            'stream_volume_L_per_batch': None,
            'stream_volume_L_per_year': None,
            'stream_component_sum_kg_per_batch': None,
            'stream_total_minus_component_sum_kg_per_batch': None,
            'include_in_is_inventory': 'yes_after_mass_mapping',
            'mapping_status': 'needs kg plastic per item and composition mapping',
            'notes': 'Used SUT is treated as an end-of-life output. Amount is usage count/area, not kg plastic yet.',
            'source_row': row['source_rows'],
            'source_column': 14,
        })
    return pd.DataFrame(records)

sut_output_rows_df = build_sut_output_rows(icr_sut_aggregated_df)
display(sut_output_rows_df[['broad_process_area','superpro_section','flow_name_superpro','amount_per_batch','unit_per_batch','include_in_is_inventory','mapping_status']])


## 9. Build the final master output inventory sheet

This sheet intentionally includes duplicate accounting views, but it labels them explicitly. The safe modelling filters are:

- for stream routing: `inventory_role == active_is_routing_mass_and_lca_treatment_candidate`, excluding product;
- for stream composition: `inventory_role == composition_metadata_only`;
- for aggregate checks: `inventory_role == overall_balance_validation_or_lca_aggregate_basis`;
- for SUT plastics: `inventory_role == sut_count_needs_mass_mapping`, then convert to kg/material composition before using in mass-based LCA or IS routing.


In [ ]:
final_columns = [
    'inventory_id',
    'source_report',
    'source_table',
    'record_type',
    'inventory_role',
    'accounting_basis',
    'broad_process_area',
    'superpro_section',
    'unit_operation',
    'area_mapping_basis',
    'source_procedure',
    'destination',
    'flow_name_superpro',
    'output_type',
    'component',
    'amount_per_batch',
    'unit_per_batch',
    'amount_per_year',
    'unit_per_year',
    'amount_per_kg_mp',
    'unit_per_kg_mp',
    'stream_volume_L_per_batch',
    'stream_volume_L_per_year',
    'stream_component_sum_kg_per_batch',
    'stream_total_minus_component_sum_kg_per_batch',
    'include_in_is_inventory',
    'mapping_status',
    'notes',
    'source_row',
    'source_column',
]

frames = [
    sr_output_stream_components_df,
    sr_output_stream_totals_df,
    sr_overall_component_outputs_df,
    sr_reconciliation_rows_df,
    sut_output_rows_df,
]

for frame in frames:
    for col in final_columns:
        if col not in frame.columns and col != 'inventory_id':
            frame[col] = None

final_output_inventory_df = pd.concat(frames, ignore_index=True, sort=False)

sort_order = {
    'output_stream_total': 1,
    'output_stream_component': 2,
    'overall_component_output': 3,
    'overall_minus_stream_detail_residual': 4,
    'sut_end_of_life_output': 5,
}
final_output_inventory_df['_sort_record_type'] = final_output_inventory_df['record_type'].map(sort_order).fillna(99)
final_output_inventory_df = final_output_inventory_df.sort_values(
    ['_sort_record_type', 'broad_process_area', 'superpro_section', 'flow_name_superpro', 'component'],
    na_position='last'
).drop(columns=['_sort_record_type']).reset_index(drop=True)

final_output_inventory_df.insert(0, 'inventory_id', [f'IS{n:04d}' for n in range(1, len(final_output_inventory_df) + 1)])
final_output_inventory_df = final_output_inventory_df[final_columns]

print('Final output inventory rows:', len(final_output_inventory_df))
display(final_output_inventory_df.head(80))


## 10. Validation checks

These checks are designed to catch exactly the issues that caused confusion before: missing WFI, section mislabels, stream total/component double counting, and residual quantities between the explicit stream detail and overall component balance.


In [ ]:
# 1. Row counts by accounting basis.
row_counts = final_output_inventory_df.groupby(['source_report', 'record_type', 'inventory_role'], dropna=False).size().reset_index(name='rows')
display(row_counts)

# 2. Stream total mass should match stream component mass up to SuperPro rounding.
stream_total_mass = sr_output_stream_totals_df['amount_per_batch'].sum()
stream_component_mass = sr_output_stream_components_df['amount_per_batch'].sum()
overall_component_mass = sr_overall_component_outputs_df['amount_per_batch'].sum()
residual_mass = sr_reconciliation_rows_df['amount_per_batch'].sum() if len(sr_reconciliation_rows_df) else 0

summary_checks = pd.DataFrame([
    {'check': 'sum output_stream_total kg/batch', 'value': stream_total_mass},
    {'check': 'sum output_stream_component kg/batch', 'value': stream_component_mass},
    {'check': 'difference stream_total - stream_component', 'value': stream_total_mass - stream_component_mass},
    {'check': 'sum overall_component_output kg/batch', 'value': overall_component_mass},
    {'check': 'sum reconciliation residual kg/batch', 'value': residual_mass},
    {'check': 'stream_component + residual kg/batch', 'value': stream_component_mass + residual_mass},
    {'check': 'overall - (stream_component + residual)', 'value': overall_component_mass - (stream_component_mass + residual_mass)},
])
display(summary_checks)

# 3. Component-by-component reconciliation.
display(reconciliation_df)

# 4. Mapping audit: there should be no review_needed area for explicit SR stream totals.
mapping_audit = final_output_inventory_df[
    (final_output_inventory_df['source_report'] == 'SR') &
    (final_output_inventory_df['record_type'].isin(['output_stream_total','output_stream_component']))
].groupby(['source_procedure','broad_process_area','superpro_section','unit_operation','area_mapping_basis'], dropna=False).size().reset_index(name='rows')
display(mapping_audit)

unmapped = mapping_audit[mapping_audit['broad_process_area'].eq('review_needed') | mapping_audit['area_mapping_basis'].eq('not_mapped')]
print('Unmapped explicit SR output rows:', len(unmapped))
if len(unmapped):
    display(unmapped)

# 5. Water/WFI visibility check.
water_rows = final_output_inventory_df[
    final_output_inventory_df['component'].astype(str).str.contains('WFI|water', case=False, na=False) |
    final_output_inventory_df['flow_name_superpro'].astype(str).str.contains('WFI|water', case=False, na=False)
]
print('Rows containing WFI/water:', len(water_rows))
display(water_rows[['inventory_id','source_report','record_type','inventory_role','broad_process_area','flow_name_superpro','component','amount_per_batch','unit_per_batch','mapping_status']])


## 11. Export the dated output workbook

The workbook has one main sheet, `final_output_inventory`, because this is the page I want to carry forward. The validation tables remain in the notebook so the exported sheet stays simple and filterable.


In [ ]:
output_xlsx = PROCESSED_DIR / f'superpro_output_inventory_master_{RUN_DATE}.xlsx'
output_csv = PROCESSED_DIR / f'superpro_output_inventory_master_{RUN_DATE}.csv'

final_output_inventory_df.to_csv(output_csv, index=False)

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    final_output_inventory_df.to_excel(writer, sheet_name='final_output_inventory', index=False)
    ws = writer.book['final_output_inventory']
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    thin = Side(style='thin', color='D9E2F3')

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = Border(bottom=thin)

    width_map = {
        'A': 12, 'B': 12, 'C': 42, 'D': 32, 'E': 48, 'F': 46, 'G': 18, 'H': 34,
        'I': 34, 'J': 36, 'K': 18, 'L': 20, 'M': 32, 'N': 34, 'O': 26, 'P': 16,
        'Q': 16, 'R': 16, 'S': 16, 'T': 18, 'U': 16, 'V': 20, 'W': 20, 'X': 24,
        'Y': 26, 'Z': 28, 'AA': 48, 'AB': 70, 'AC': 16, 'AD': 14,
    }
    for col_letter, width in width_map.items():
        ws.column_dimensions[col_letter].width = width

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical='top', wrap_text=True)

    numeric_cols = ['P', 'R', 'T', 'V', 'W', 'X', 'Y']
    for col in numeric_cols:
        for cell in ws[col][1:]:
            cell.number_format = '0.0000'

print('Wrote Excel:', output_xlsx)
print('Wrote CSV:', output_csv)
